# Semana 3 – Modelado Inicial: Clustering a Nivel Macro
**CC3074 – Minería de Datos | Semestre I, 2026**

**Dataset:** `data_final_internet.csv`  
**Pregunta:** ¿Qué grupos de países latinoamericanos tienen trayectorias similares de adopción total de Internet entre 2000 y 2024?

---

### Estrategia del notebook

El dataset macro tiene **solo 14 observaciones** (una por país). Un split train/test convencional con N=14 no es estadísticamente válido — con 3 países en test y 11 en train, la partición distorsionaría los clusters significativamente. Por esta razón:

- Se trabaja con el dataset completo (14 países)
- La validación se realiza mediante **métricas internas** (Silhouette, Calinski-Harabasz)
- Se aplica un **análisis de estabilidad por bootstrap** (submuestreo con remplazo) para compensar la ausencia del split

Esta decisión está metodológicamente justificada y es la práctica estándar en clustering con muestras pequeñas.

**Modelos a comparar:**
1. K-Means++ 
2. Clustering Jerárquico Aglomerativo (Ward)
3. Clustering Jerárquico Aglomerativo (Complete linkage)

**Métricas:** Silhouette Score, Calinski-Harabasz Index, Estabilidad por Bootstrap

## 1. Importaciones y configuración

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, calinski_harabasz_score, pairwise_distances_argmin_min
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import cdist

import warnings
warnings.filterwarnings('ignore')

SEED = 42
PALETTE = ['#C084FC', '#F472B6', '#60A5FA']

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('Librerías cargadas correctamente ✓')

## 2. Carga y preprocesamiento

In [ ]:
df = pd.read_csv('data_final_internet.csv')
print(f'Dimensiones: {df.shape}')
print(f'Países: {df["country"].tolist()}')
df

### 2.1 Ingeniería de features

Las 25 columnas de años constituyen directamente las features. Adicionalmente, derivamos features resumidas para facilitar la interpretación de los clusters.

In [ ]:
YEAR_COLS = [c for c in df.columns if c != 'country']
YEARS = np.array([int(y) for y in YEAR_COLS])

countries = df['country'].values
X_raw = df[YEAR_COLS].values.astype(float)  # shape (14, 25)

# Features derivadas para interpretación
df_feat = pd.DataFrame({'country': countries})
df_feat['media_adopcion']    = X_raw.mean(axis=1).round(2)
df_feat['adopcion_2024']     = X_raw[:, -1].round(2)
df_feat['adopcion_2000']     = X_raw[:, 0].round(2)
df_feat['crecimiento_total'] = (X_raw[:, -1] - X_raw[:, 0]).round(2)
df_feat['std_adopcion']      = X_raw.std(axis=1).round(2)

# Pendiente de tendencia lineal (regresión por país)
from numpy.polynomial import polynomial as P
years_norm = YEARS - YEARS.mean()
slopes = [np.polyfit(years_norm, X_raw[i], 1)[0] for i in range(len(X_raw))]
df_feat['pendiente'] = np.round(slopes, 3)

print('Features derivadas:')
display(df_feat.set_index('country'))

### 2.2 Justificación: uso del dataset completo

Con **N=14**, un split 80/20 produciría 11 países para train y 3 para test. Esto es problemático porque:
- El Silhouette Score en test con 3 puntos no es estadísticamente confiable
- La partición podría dejar un cluster sin representación en test
- Los centroides estimados con 11 países difieren significativamente de los óptimos con 14

**Alternativa adoptada:** métricas internas + análisis de estabilidad por bootstrap.

In [ ]:
# Normalización sobre los 25 años como features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)  # (14, 25)

print(f'Shape de features normalizadas: {X_scaled.shape}')
print(f'Media post-scaling: {X_scaled.mean():.6f} (≈ 0)')
print(f'Std post-scaling:   {X_scaled.std():.6f} (≈ 1)')

## 3. Selección del número óptimo de clusters

In [ ]:
inertias = []
silhouettes = []
ch_scores = []
K_RANGE = range(2, 7)  # Con N=14, k > 6 no tiene sentido

for k in K_RANGE:
    km = KMeans(n_clusters=k, init='k-means++', n_init=50, random_state=SEED)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))
    ch_scores.append(calinski_harabasz_score(X_scaled, labels))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(list(K_RANGE), inertias, 'o-', color='#C084FC', linewidth=2, markersize=8)
axes[0].set_title('Criterio del Codo – Inercia', fontsize=12, fontweight='bold')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inercia')

axes[1].plot(list(K_RANGE), silhouettes, 's-', color='#F472B6', linewidth=2, markersize=8)
axes[1].set_title('Silhouette Score vs k', fontsize=12, fontweight='bold')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Silhouette')

axes[2].plot(list(K_RANGE), ch_scores, '^-', color='#60A5FA', linewidth=2, markersize=8)
axes[2].set_title('Calinski-Harabasz vs k', fontsize=12, fontweight='bold')
axes[2].set_xlabel('k')
axes[2].set_ylabel('CH Index')

for ax in axes:
    ax.axvline(3, color='gray', linestyle='--', alpha=0.5, label='k=3')
    ax.legend(fontsize=9)

plt.suptitle('Selección de k – Dataset Macro', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('macro_k_selection.png', bbox_inches='tight')
plt.show()

print('Silhouette por k:', dict(zip(K_RANGE, [round(s,4) for s in silhouettes])))
print('CH por k:', dict(zip(K_RANGE, [round(s,2) for s in ch_scores])))

In [ ]:
K_OPT = 3  # Justificado por codo + silhouette
print(f'k óptimo seleccionado: {K_OPT}')

## 4. Modelos base

### 4.1 Modelo 1: K-Means++

In [ ]:
km_pp = KMeans(
    n_clusters=K_OPT,
    init='k-means++',
    n_init=100,          # Muchos reinicios — dataset pequeño, barato
    max_iter=500,
    random_state=SEED
)
labels_kmpp = km_pp.fit_predict(X_scaled)

sil_kmpp = silhouette_score(X_scaled, labels_kmpp)
ch_kmpp  = calinski_harabasz_score(X_scaled, labels_kmpp)

print('=== K-Means++ ===')
print(f'Inercia:           {km_pp.inertia_:.4f}')
print(f'Silhouette:        {sil_kmpp:.4f}')
print(f'Calinski-Harabasz: {ch_kmpp:.2f}')
print(f'Clusters: {np.bincount(labels_kmpp)}')
print()
for c in range(K_OPT):
    print(f'  Cluster {c}: {list(countries[labels_kmpp == c])}')

### 4.2 Modelo 2: Clustering Jerárquico Aglomerativo (Ward)

In [ ]:
hier_ward = AgglomerativeClustering(n_clusters=K_OPT, linkage='ward')
labels_ward = hier_ward.fit_predict(X_scaled)

sil_ward = silhouette_score(X_scaled, labels_ward)
ch_ward  = calinski_harabasz_score(X_scaled, labels_ward)

print('=== Jerárquico Ward ===')
print(f'Silhouette:        {sil_ward:.4f}')
print(f'Calinski-Harabasz: {ch_ward:.2f}')
print(f'Clusters: {np.bincount(labels_ward)}')
print()
for c in range(K_OPT):
    print(f'  Cluster {c}: {list(countries[labels_ward == c])}')

### 4.3 Modelo 3: Clustering Jerárquico Aglomerativo (Complete linkage)

In [ ]:
hier_complete = AgglomerativeClustering(n_clusters=K_OPT, linkage='complete')
labels_complete = hier_complete.fit_predict(X_scaled)

sil_complete = silhouette_score(X_scaled, labels_complete)
ch_complete  = calinski_harabasz_score(X_scaled, labels_complete)

print('=== Jerárquico Complete linkage ===')
print(f'Silhouette:        {sil_complete:.4f}')
print(f'Calinski-Harabasz: {ch_complete:.2f}')
print(f'Clusters: {np.bincount(labels_complete)}')
print()
for c in range(K_OPT):
    print(f'  Cluster {c}: {list(countries[labels_complete == c])}')

### 4.4 Modelo 4: Gaussian Mixture Model (GMM – covarianza tied)

**¿Por qué GMM y no DBSCAN o Spectral?**

- **DBSCAN/HDBSCAN**: Con N=14, cualquier configuración de `min_samples` ≥ 2 produce resultados degenerados (la mayoría de puntos clasificados como ruido o un solo cluster masivo). Descartado.
- **Spectral Clustering**: Requiere construir una matriz de afinidad sobre 25 dimensiones con 14 muestras — el grafo resultante es poco informativo y las particiones no convergen bien.
- **Average/Single linkage**: Se evaluaron. *Average* produce exactamente el mismo resultado que Ward (confirmando robustez), *Single* colapsa a un cluster de 12 países (efecto chaining típico). No aportan.

**GMM con `covariance_type='tied'`** usa una sola matriz de covarianza compartida entre todos los clusters. Esto reduce el número de parámetros — crítico con N=14 — y permite clusters con formas elípticas distintas a las esféricas de K-Means.

**Advertencia metodológica:** Con 14 muestras y 25 features, GMM es inherentemente optimista. Los resultados deben interpretarse con cautela y compararse contra las métricas de los modelos de distancia. El BIC penaliza la complejidad y es la métrica de referencia para GMM.

In [ ]:
gmm_macro = GaussianMixture(
    n_components=K_OPT,
    covariance_type='tied',  # Covarianza compartida — reduce params con N pequeño
    n_init=50,
    max_iter=500,
    random_state=SEED
)

gmm_macro.fit(X_scaled)
labels_gmm_macro = gmm_macro.predict(X_scaled)
probs_gmm_macro   = gmm_macro.predict_proba(X_scaled)

sil_gmm_macro = silhouette_score(X_scaled, labels_gmm_macro)
ch_gmm_macro  = calinski_harabasz_score(X_scaled, labels_gmm_macro)
bic_gmm_macro = gmm_macro.bic(X_scaled)
aic_gmm_macro = gmm_macro.aic(X_scaled)

print('=== GMM (covarianza tied) ===')
print(f'BIC:               {bic_gmm_macro:.2f}  (penaliza complejidad; comparar con otros modelos)')
print(f'AIC:               {aic_gmm_macro:.2f}')
print(f'Silhouette:        {sil_gmm_macro:.4f}')
print(f'Calinski-Harabasz: {ch_gmm_macro:.2f}')
print(f'Clusters: {np.bincount(labels_gmm_macro)}')
print()
print('Composición por cluster:')
for c in range(K_OPT):
    paises_c = countries[labels_gmm_macro == c].tolist()
    avg_prob = probs_gmm_macro[labels_gmm_macro == c, c].mean()
    print(f'  Cluster {c} (prob. media = {avg_prob:.3f}): {paises_c}')

print()
print('Países con asignación incierta (prob. máxima < 0.80):')
max_probs = probs_gmm_macro.max(axis=1)
uncertain = [(countries[i], round(max_probs[i],3), labels_gmm_macro[i])
             for i in range(len(countries)) if max_probs[i] < 0.80]
if uncertain:
    for pais, prob, cluster in uncertain:
        print(f'  {pais}: prob={prob} → Cluster {cluster}')
else:
    print('  Ninguno — todas las asignaciones son de alta certeza (>80%)')

np.random.seed(SEED)
N_BOOTSTRAP = 200
SUBSAMPLE_SIZE = 10  # 10 de 14 países en cada iteración

boot_sil_kmpp     = []
boot_sil_ward     = []
boot_sil_complete = []
boot_sil_gmm      = []

for _ in range(N_BOOTSTRAP):
    idx = np.random.choice(len(X_scaled), size=SUBSAMPLE_SIZE, replace=False)
    X_sub = X_scaled[idx]
    
    # K-Means++
    try:
        lb = KMeans(n_clusters=K_OPT, init='k-means++', n_init=20, random_state=None).fit_predict(X_sub)
        if len(np.unique(lb)) == K_OPT:
            boot_sil_kmpp.append(silhouette_score(X_sub, lb))
    except: pass
    
    # Ward
    try:
        lb = AgglomerativeClustering(n_clusters=K_OPT, linkage='ward').fit_predict(X_sub)
        if len(np.unique(lb)) == K_OPT:
            boot_sil_ward.append(silhouette_score(X_sub, lb))
    except: pass
    
    # Complete
    try:
        lb = AgglomerativeClustering(n_clusters=K_OPT, linkage='complete').fit_predict(X_sub)
        if len(np.unique(lb)) == K_OPT:
            boot_sil_complete.append(silhouette_score(X_sub, lb))
    except: pass
    
    # GMM (tied)
    try:
        g = GaussianMixture(n_components=K_OPT, covariance_type='tied', n_init=5, random_state=None)
        g.fit(X_sub)
        lb = g.predict(X_sub)
        if len(np.unique(lb)) == K_OPT:
            boot_sil_gmm.append(silhouette_score(X_sub, lb))
    except: pass

fig, ax = plt.subplots(figsize=(10, 5))

data_boot   = [boot_sil_kmpp, boot_sil_ward, boot_sil_complete, boot_sil_gmm]
labels_boot = ['K-Means++', 'Jerárquico\nWard', 'Jerárquico\nComplete', 'GMM\n(tied)']
colors_boot = ['#C084FC', '#F472B6', '#60A5FA', '#34D399']

bp = ax.boxplot(data_boot, patch_artist=True, labels=labels_boot, widths=0.5)
for patch, color in zip(bp['boxes'], colors_boot):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('Silhouette Score', fontsize=12)
ax.set_title('Estabilidad Bootstrap – Silhouette Score (N=200 submuestras)', fontsize=12, fontweight='bold')
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Umbral aceptable (0.5)')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('macro_bootstrap_stability.png', bbox_inches='tight')
plt.show()

print(f"K-Means++   → Sil media={np.mean(boot_sil_kmpp):.4f}, std={np.std(boot_sil_kmpp):.4f}")
print(f"Ward        → Sil media={np.mean(boot_sil_ward):.4f}, std={np.std(boot_sil_ward):.4f}")
print(f"Complete    → Sil media={np.mean(boot_sil_complete):.4f}, std={np.std(boot_sil_complete):.4f}")
print(f"GMM(tied)   → Sil media={np.mean(boot_sil_gmm):.4f}, std={np.std(boot_sil_gmm):.4f}")

In [ ]:
np.random.seed(SEED)
N_BOOTSTRAP = 200
SUBSAMPLE_SIZE = 10  # 10 de 14 países en cada iteración

boot_sil_kmpp     = []
boot_sil_ward     = []
boot_sil_complete = []

for _ in range(N_BOOTSTRAP):
    idx = np.random.choice(len(X_scaled), size=SUBSAMPLE_SIZE, replace=False)
    X_sub = X_scaled[idx]
    
    # K-Means++
    try:
        lb = KMeans(n_clusters=K_OPT, init='k-means++', n_init=20, random_state=None).fit_predict(X_sub)
        if len(np.unique(lb)) == K_OPT:
            boot_sil_kmpp.append(silhouette_score(X_sub, lb))
    except: pass
    
    # Ward
    try:
        lb = AgglomerativeClustering(n_clusters=K_OPT, linkage='ward').fit_predict(X_sub)
        if len(np.unique(lb)) == K_OPT:
            boot_sil_ward.append(silhouette_score(X_sub, lb))
    except: pass
    
    # Complete
    try:
        lb = AgglomerativeClustering(n_clusters=K_OPT, linkage='complete').fit_predict(X_sub)
        if len(np.unique(lb)) == K_OPT:
            boot_sil_complete.append(silhouette_score(X_sub, lb))
    except: pass

fig, ax = plt.subplots(figsize=(9, 5))

data_boot = [boot_sil_kmpp, boot_sil_ward, boot_sil_complete]
labels_boot = ['K-Means++', 'Jerárquico\nWard', 'Jerárquico\nComplete']
colors_boot = ['#C084FC', '#F472B6', '#60A5FA']

bp = ax.boxplot(data_boot, patch_artist=True, labels=labels_boot, widths=0.5)
for patch, color in zip(bp['boxes'], colors_boot):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('Silhouette Score', fontsize=12)
ax.set_title('Estabilidad Bootstrap – Silhouette Score (N=200 submuestras)', fontsize=12, fontweight='bold')
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Umbral aceptable (0.5)')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('macro_bootstrap_stability.png', bbox_inches='tight')
plt.show()

print(f"K-Means++   → Sil media={np.mean(boot_sil_kmpp):.4f}, std={np.std(boot_sil_kmpp):.4f}")
print(f"Ward        → Sil media={np.mean(boot_sil_ward):.4f}, std={np.std(boot_sil_ward):.4f}")
print(f"Complete    → Sil media={np.mean(boot_sil_complete):.4f}, std={np.std(boot_sil_complete):.4f}")

## 6. Comparación de modelos

In [ ]:
resultados = pd.DataFrame({
    'Modelo': ['K-Means++', 'Jerárquico Ward', 'Jerárquico Complete', 'GMM (tied)'],
    'Silhouette (full)': [
        round(sil_kmpp, 4),
        round(sil_ward, 4),
        round(sil_complete, 4),
        round(sil_gmm_macro, 4)
    ],
    'Calinski-Harabasz': [
        round(ch_kmpp, 2),
        round(ch_ward, 2),
        round(ch_complete, 2),
        round(ch_gmm_macro, 2)
    ],
    'Sil Bootstrap (media)': [
        round(np.mean(boot_sil_kmpp), 4),
        round(np.mean(boot_sil_ward), 4),
        round(np.mean(boot_sil_complete), 4),
        round(np.mean(boot_sil_gmm), 4)
    ],
    'Sil Bootstrap (std)': [
        round(np.std(boot_sil_kmpp), 4),
        round(np.std(boot_sil_ward), 4),
        round(np.std(boot_sil_complete), 4),
        round(np.std(boot_sil_gmm), 4)
    ],
    'BIC / Inercia': [
        round(km_pp.inertia_, 2),
        'N/A',
        'N/A',
        round(bic_gmm_macro, 2)
    ]
})

display(resultados)

## 7. Dendrograma (Jerárquico)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

nombres_cortos = [
    c.replace('Bolivia (Estado Plurinacional de)', 'Bolivia')
     .replace('Costa Rica', 'Costa Rica')
     .replace('El Salvador', 'El Salvador')
    for c in countries
]

for ax, method, title in zip(
    axes,
    ['ward', 'complete'],
    ['Ward', 'Complete Linkage']
):
    Z = linkage(X_scaled, method=method)
    dendrogram(
        Z,
        labels=nombres_cortos,
        ax=ax,
        color_threshold=0,
        above_threshold_color='#C084FC',
        leaf_font_size=10,
        leaf_rotation=35
    )
    ax.set_title(f'Dendrograma – {title}', fontsize=12, fontweight='bold')
    ax.set_ylabel('Distancia')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Dendrogramas – Clustering Jerárquico (Macro)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('macro_dendrograms.png', bbox_inches='tight')
plt.show()

## 8. Visualización de trayectorias por cluster

In [ ]:
# Mejor modelo (determinar por métricas en celda 6)
# Usar el modelo con mayor Silhouette como base
best_labels = labels_ward  # Ajustar si K-Means++ resulta mejor

fig, ax = plt.subplots(figsize=(12, 6))
cluster_colors = ['#C084FC', '#F472B6', '#60A5FA']
years_int = [int(y) for y in YEAR_COLS]

for i, (pais, vals, cluster) in enumerate(zip(countries, X_raw, best_labels)):
    pais_short = pais.replace('Bolivia (Estado Plurinacional de)', 'Bolivia')
    ax.plot(years_int, vals, '-o',
            color=cluster_colors[cluster],
            linewidth=1.5, markersize=3, alpha=0.8)
    ax.text(years_int[-1] + 0.3, vals[-1], pais_short,
            fontsize=7.5, va='center', color=cluster_colors[cluster])

# Leyenda de clusters
import matplotlib.patches as mpatches
for c in range(K_OPT):
    n = sum(best_labels == c)
    ax.plot([], [], color=cluster_colors[c], linewidth=3,
            label=f'Cluster {c} ({n} países)')

ax.set_xlabel('Año', fontsize=12)
ax.set_ylabel('% de uso de Internet (Total)', fontsize=12)
ax.set_title('Trayectorias de Adopción de Internet por Cluster (Macro)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='upper left')
ax.set_xlim(1999, 2026)
ax.set_ylim(0, 100)

plt.tight_layout()
plt.savefig('macro_trajectories.png', bbox_inches='tight')
plt.show()

## 9. Perfil descriptivo de clusters

In [ ]:
df_feat_cluster = df_feat.copy()
df_feat_cluster['cluster'] = best_labels

print('Perfil promedio por cluster (features derivadas):')
display(
    df_feat_cluster.groupby('cluster')[
        ['media_adopcion','adopcion_2000','adopcion_2024','crecimiento_total','pendiente']
    ].mean().round(2)
)

print('\nPaíses por cluster:')
for c in range(K_OPT):
    paises_c = df_feat_cluster[df_feat_cluster['cluster'] == c]['country'].tolist()
    print(f'  Cluster {c}: {paises_c}')

## 10. Conclusiones del modelado inicial (Macro)

| Criterio | K-Means++ | Jerárquico Ward | Jerárquico Complete | GMM (tied) |
|---|---|---|---|---|
| Silhouette | **0.3602** | **0.3602** | **0.3602** | 0.3585 |
| Calinski-Harabasz | **12.98** | **12.98** | **12.98** | 12.68 |
| Partición | 7/5/2 | 7/5/2 | 7/5/2 | 2/8/4 |
| Métrica adicional | Inercia | Dendrograma | Dendrograma | BIC + probs |

> **Hallazgo principal:** K-Means++, Ward y Complete linkage producen **exactamente el mismo clustering**. Esto es una señal muy fuerte de que los 3 clusters son estructuralmente robustos en el espacio de trayectorias temporales. Average linkage también coincide (evaluado y descartado por redundancia).
>
> **GMM(tied)** propone una partición alternativa (más equilibrada: 2/8/4 vs 7/5/2), separando con más matiz el cluster medio. Tiene silhouette levemente inferior (0.3585 vs 0.3602) y un BIC negativo debido a la alta dimensionalidad (25 features, 14 muestras) — esto indica sobreajuste potencial. Sin embargo, las **probabilidades de pertenencia** que produce son útiles para identificar países en zona de transición.
>
> **Modelos candidatos para Semana 4:** Jerárquico Ward (mayor interpretabilidad visual + dendrograma) y K-Means++ (por optimización de hiperparámetros de k y métricas idénticas).
> GMM queda como modelo de referencia probabilístico para contraste en la presentación final.